# Interactive ML Web App – Titanic Survival Prediction

**Project:** Deploy a trained Decision Tree classifier as an interactive Gradio web app.

Users can change passenger details using sliders and dropdowns, then receive a live survival prediction and estimated probability.

**Run in Google Colab:** Execute cells from top to bottom. The final cell launches a public temporary Gradio link.

> This is a learning demonstration using the Titanic dataset. Predictions are not real-world advice.

## 1. Install Gradio

In [ ]:
!pip -q install gradio

## 2. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Libraries imported successfully!")

## 3. Load the Titanic dataset

In [ ]:
df = sns.load_dataset("titanic")

print("Dataset shape:", df.shape)
display(df.head())

## 4. Prepare features and target

In [ ]:
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
target = "survived"

X = df[features].copy()
y = df[target].copy()

numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked"]

print("Features:", features)
print("Target:", target)

## 5. Split training and testing data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 6. Train the Decision Tree model

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=4,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model trained successfully! Test accuracy: {accuracy:.2%}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Did not survive", "Survived"]))

## 7. Create the prediction function

In [ ]:
def predict_survival(pclass, sex, age, sibsp, parch, fare, embarked):
    # Build a one-row DataFrame with the same feature names used during training.
    passenger = pd.DataFrame([{
        "pclass": int(pclass),
        "sex": sex,
        "age": float(age),
        "sibsp": int(sibsp),
        "parch": int(parch),
        "fare": float(fare),
        "embarked": embarked
    }])

    prediction = int(model.predict(passenger)[0])
    probabilities = model.predict_proba(passenger)[0]

    labels = {0: "Did not survive", 1: "Survived"}
    confidence = float(probabilities[prediction])

    result = (
        f"### Prediction: {labels[prediction]}\n"
        f"**Estimated model probability for this prediction:** {confidence:.1%}\n\n"
        f"**Model test accuracy:** {accuracy:.1%}"
    )

    probability_output = {
        "Did not survive": float(probabilities[0]),
        "Survived": float(probabilities[1])
    }

    return result, probability_output

print("Prediction function is ready!")

## 8. Build the interactive Gradio app

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="Titanic Survival Predictor") as demo:
    gr.Markdown(
        "# 🚢 Titanic Survival Predictor\n"
        "Change the passenger details below and click **Predict Survival**. "
        "The trained Decision Tree model will return a prediction and class probabilities."
    )

    with gr.Row():
        with gr.Column():
            pclass = gr.Dropdown(
                choices=[1, 2, 3],
                value=3,
                label="Passenger class"
            )
            sex = gr.Radio(
                choices=["female", "male"],
                value="male",
                label="Gender"
            )
            age = gr.Slider(
                minimum=0, maximum=80, value=25, step=1,
                label="Age"
            )
            sibsp = gr.Slider(
                minimum=0, maximum=8, value=0, step=1,
                label="Siblings / spouses aboard"
            )
            parch = gr.Slider(
                minimum=0, maximum=6, value=0, step=1,
                label="Parents / children aboard"
            )
            fare = gr.Slider(
                minimum=0, maximum=513, value=7.25, step=0.25,
                label="Ticket fare"
            )
            embarked = gr.Dropdown(
                choices=["S", "C", "Q"],
                value="S",
                label="Port of embarkation (S, C, Q)"
            )
            predict_button = gr.Button("Predict Survival", variant="primary")

        with gr.Column():
            prediction_output = gr.Markdown(label="Prediction")
            probability_output = gr.Label(
                num_top_classes=2,
                label="Model class probabilities"
            )

    predict_button.click(
        fn=predict_survival,
        inputs=[pclass, sex, age, sibsp, parch, fare, embarked],
        outputs=[prediction_output, probability_output]
    )

    gr.Markdown(
        f"**Model:** Decision Tree Classifier  |  "
        f"**Held-out test accuracy:** {accuracy:.1%}\n\n"
        "Educational demonstration only. Model probabilities are estimates, "
        "not guarantees or causal explanations."
    )

print("Gradio app interface created!")

## 9. Launch the web app

In [ ]:
# share=True creates a temporary public link that you can open in a browser.
# Keep this Colab cell running while you use the app.
demo.launch(share=True, debug=True)

## 10. Try sample passenger inputs

After launching the app, try changing:
- Passenger class from 3 to 1
- Gender from male to female
- Age from 25 to 8
- Fare from 7.25 to 80

Click **Predict Survival** after changing the inputs and compare the output probabilities.

## Conclusion

This project demonstrates a complete beginner deployment workflow: load a dataset, preprocess data, train a Scikit-Learn Decision Tree, evaluate it on a held-out test set, and connect it to a Gradio interface for interactive predictions.

**Deployment note:** The `share=True` link is temporary and works while the Colab runtime and launch cell remain active. It is not permanent hosting. For a permanent public deployment, publish the app to Hugging Face Spaces or another hosting provider.